In [ ]:
# ! pip install numpy 

### Import the necessary libraries

In [ ]:
!pip install torch

In [ ]:
import numpy as np

### Neural Network Class

### Define the dataset

In [ ]:
# Define dataset, classify gender
data = np.array([
  [-20, -20],  # Alice
  [17, 50],   # Bob
  [17, 53],   # Charlie
  [-10, -15], 
])
all_y_trues = np.array([
  0, 
  1, 
  1, 
  0,
])

### Train our neural network!

### Make some predictions

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# Define the model
model = Sequential()
# Hidden layer with 2 neurons, input_dim=2 as we have 2 input features
model.add(Dense(units=8, input_dim=2, activation='softmax'))

# Output layer with 1 neuron
model.add(Dense(units=1, activation='relu'))

# Compile the model
model.compile(optimizer=SGD(learning_rate=0.5), loss='mean_squared_error')

# Example data and labels (replace this with your actual data)
data = np.array([[-20, -20], [17, 50], [17, 53], [-10, -15]])
all_y_trues = np.array([0, 1, 1, 0])

# Train the model
model.fit(data, all_y_trues, epochs=50, verbose=1)

In [ ]:
test = np.array([20,30])
prediction = model.predict(test.reshape(1,2))
prediction

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Define the dataset
data = torch.tensor([
    [-20, -20],  # Alice
    [17, 50],    # Bob
    [17, 53],    # Charlie
    [-10, -15]
], dtype=torch.float32)

all_y_trues = torch.tensor([0, 1, 1, 0], dtype=torch.float32).unsqueeze(1)  # Add extra dimension for MSELoss

# Define the Neural Network
class NeuralNetwork(nn.Module):
    
    #Intialize the object model = NeuralNetwork()
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.hidden = nn.Linear(2, 8)  # Input layer with 2 inputs, 2 outputs       
        self.output = nn.Linear(8, 1)  # Output layer with 1 neuron

    def forward(self, x):
        x = torch.softmax(self.hidden(x), dim=1)  # Apply softmax to hidden layer
        x = torch.relu(self.output(x))  # Apply ReLU to output layer
        return x

model = NeuralNetwork()

# Define the loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# Wrap data into a DataLoader
dataset = TensorDataset(data, all_y_trues)
train_dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Training loop
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

        
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 1 == 0:  # Log every batch
            print(f"loss: {loss.item():>7f}  [{batch * len(X):>5d}/{size:>5d}]")


In [ ]:
# pytorch-lightning #TorchMetrics
# huggingface - early stopping

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ShadowNeuralNetwork(nn.Module):
    def __init__(self, decay_rate=0.01, shadow_strength=0.01):
        super(ShadowNeuralNetwork, self).__init__()
        # Main network layers
        self.hidden = nn.Linear(2, 8)
        self.output = nn.Linear(8, 1)
        
        # Initialize shadow matrices with small random values
        self.register_buffer('hidden_shadow', torch.randn(2, 8) * 0.01)
        self.register_buffer('output_shadow', torch.randn(8, 1) * 0.01)
        
        # Hyperparameters
        self.decay_rate = decay_rate
        self.shadow_strength = shadow_strength
        
    def update_shadow(self, layer_input, layer_output, shadow):
        with torch.no_grad():
            # Normalize inputs to prevent explosion
            layer_input = F.normalize(layer_input, dim=1)
            layer_output = F.normalize(layer_output, dim=1)
            
            # Compute activation pattern
            activation_pattern = torch.matmul(layer_input.t(), layer_output)
            
            # Normalize activation pattern
            activation_pattern = F.normalize(activation_pattern, dim=1)
            
            # Update shadow with decay
            new_shadow = (1 - self.decay_rate) * shadow + self.decay_rate * activation_pattern
            
            # Normalize shadow to prevent accumulation
            new_shadow = F.normalize(new_shadow, dim=1)
            
            return new_shadow

    def forward(self, x):
        # Forward pass through hidden layer
        hidden_pre = self.hidden(x)
        hidden_out = torch.tanh(hidden_pre)  # Using tanh instead of softmax for stability
        
        # Update hidden layer shadow matrix
        self.hidden_shadow = self.update_shadow(x, hidden_out, self.hidden_shadow)
        
        # Apply scaled shadow influence
        shadow_influence = self.shadow_strength * torch.matmul(x, self.hidden_shadow)
        hidden_with_shadow = hidden_out + shadow_influence
        
        # Normalize combined output
        hidden_with_shadow = F.normalize(hidden_with_shadow, dim=1)
        
        # Forward pass through output layer
        output = torch.sigmoid(self.output(hidden_with_shadow))  # Using sigmoid for binary classification
        
        # Update output layer shadow matrix
        self.output_shadow = self.update_shadow(hidden_with_shadow, output, self.output_shadow)
        
        return output

# Data setup
data = torch.tensor([
    [-20, -20],  # Alice
    [17, 50],    # Bob
    [17, 53],    # Charlie
    [-10, -15]
], dtype=torch.float32)

# Normalize input data
data = F.normalize(data, dim=1)

all_y_trues = torch.tensor([0, 1, 1, 0], dtype=torch.float32).unsqueeze(1)

# Initialize model and training components
model = ShadowNeuralNetwork()
loss_fn = nn.BCELoss()  # Binary Cross Entropy for binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Using Adam optimizer

def train(epochs=1000):
    losses = []
    for epoch in range(epochs):
        # Forward pass
        optimizer.zero_grad()
        pred = model(data)
        loss = loss_fn(pred, all_y_trues)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        if epoch % 100 == 0:
            with torch.no_grad():
                print(f'Epoch {epoch}, Loss: {loss.item():.4f}')
                print(f'Shadow matrix influence: {model.hidden_shadow.mean().item():.4f}')
                print(f'Predictions: {pred.squeeze().tolist()}\n')
        
        losses.append(loss.item())
    
    return losses

# Run training
losses = train()

# Make predictions
with torch.no_grad():
    final_predictions = model(data)
    print("\nFinal Predictions:")
    print(final_predictions.squeeze().tolist())

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F

# Define the ShadowNeuralNetwork with adjustable input size
class ShadowNeuralNetwork(nn.Module):
    def __init__(self, input_size=2, hidden_size=8, decay_rate=0.01, shadow_strength=0.01):
        super(ShadowNeuralNetwork, self).__init__()
        # Main network layers
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, 1)
        
        # Initialize shadow matrices with small random values
        self.register_buffer('hidden_shadow', torch.randn(input_size, hidden_size) * 0.01)
        self.register_buffer('output_shadow', torch.randn(hidden_size, 1) * 0.01)
        
        # Hyperparameters
        self.decay_rate = decay_rate
        self.shadow_strength = shadow_strength
        
    def update_shadow(self, layer_input, layer_output, shadow):
        with torch.no_grad():
            # Normalize inputs to prevent explosion
            layer_input = F.normalize(layer_input, dim=1)
            layer_output = F.normalize(layer_output, dim=1)
            
            # Compute activation pattern
            activation_pattern = torch.matmul(layer_input.t(), layer_output)
            
            # Normalize activation pattern
            activation_pattern = F.normalize(activation_pattern, dim=1)
            
            # Update shadow with decay
            new_shadow = (1 - self.decay_rate) * shadow + self.decay_rate * activation_pattern
            
            # Normalize shadow to prevent accumulation
            new_shadow = F.normalize(new_shadow, dim=1)
            
            return new_shadow

    def forward(self, x):
        # Forward pass through hidden layer
        hidden_pre = self.hidden(x)
        hidden_out = torch.tanh(hidden_pre)  # Using tanh instead of softmax for stability
        
        # Update hidden layer shadow matrix
        self.hidden_shadow = self.update_shadow(x, hidden_out, self.hidden_shadow)
        
        # Apply scaled shadow influence
        shadow_influence = self.shadow_strength * torch.matmul(x, self.hidden_shadow)
        hidden_with_shadow = hidden_out + shadow_influence
        
        # Normalize combined output
        hidden_with_shadow = F.normalize(hidden_with_shadow, dim=1)
        
        # Forward pass through output layer
        output = torch.sigmoid(self.output(hidden_with_shadow))  # Using sigmoid for binary classification
        
        # Update output layer shadow matrix
        self.output_shadow = self.update_shadow(hidden_with_shadow, output, self.output_shadow)
        
        return output

# Generate synthetic data
X, y = make_classification(
    n_samples=1000,    # Number of samples
    n_features=20,     # Number of features
    n_informative=2,   # Number of informative features
    n_redundant=2,     # Number of redundant features
    n_clusters_per_class=1,
    flip_y=0.01,       # Label noise
    class_sep=1.0,
    random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create custom Dataset
class SyntheticDataset(Dataset):
    def __init__(self, features, labels):
        self.X = features
        self.y = labels
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Instantiate datasets
train_dataset = SyntheticDataset(X_train_tensor, y_train_tensor)
test_dataset = SyntheticDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize model and training components
input_size = X_train.shape[1]  # Number of features (20)
model = ShadowNeuralNetwork(input_size=input_size)  # Set input_size=20
loss_fn = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
def train_model(model, train_loader, loss_fn, optimizer, epochs=1000):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X_batch.size(0)
        
        epoch_loss /= len(train_loader.dataset)
        
        if (epoch + 1) % 100 == 0 or epoch == 0:
            # Calculate shadow influence mean
            shadow_influence_mean = model.hidden_shadow.mean().item()
            print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')
            print(f'Shadow matrix influence mean: {shadow_influence_mean:.4f}')
            
            # Make predictions on the entire training set for logging
            with torch.no_grad():
                train_preds = model(X_train_tensor)
                train_preds = train_preds.squeeze().tolist()
                print(f'Predictions Sample: {train_preds[:4]}\n')

# Train the model
train_model(model, train_loader, loss_fn, optimizer, epochs=500)

# Evaluate the model
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted = (predictions >= 0.5).float()
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()
    accuracy = correct / total
    print(f'Accuracy on test set: {accuracy*100:.2f}%')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_shadow_matrix(shadow_matrix, epoch):
    shadow = shadow_matrix.cpu().numpy()
    plt.figure(figsize=(10, 8))
    sns.heatmap(shadow, cmap='viridis')
    plt.title(f'Shadow Matrix at Epoch {epoch}')
    plt.xlabel('Hidden Units')
    plt.ylabel('Input Features')
    plt.show()

# Example usage after certain epochs
visualize_shadow_matrix(model.hidden_shadow, epoch=500)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        all_preds.extend(predictions.squeeze().tolist())
        all_labels.extend(y_batch.squeeze().tolist())

# Convert probabilities to binary predictions
binary_preds = [1.0 if p >= 0.5 else 0.0 for p in all_preds]

# Calculate metrics
accuracy = accuracy_score(all_labels, binary_preds)
precision = precision_score(all_labels, binary_preds)
recall = recall_score(all_labels, binary_preds)
f1 = f1_score(all_labels, binary_preds)
roc_auc = roc_auc_score(all_labels, all_preds)

print(f'Accuracy on test set: {accuracy*100:.2f}%')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')


In [ ]:
# Train the mdel

train()


In [ ]:
from sklearn.manifold import TSNE

# Extract hidden layer activations with and without shadow influence
hidden_without_shadow = torch.tanh(model.hidden(X_test_tensor)).cpu().detach().numpy()
hidden_with_shadow = F.normalize(
    torch.tanh(model.hidden(X_test_tensor)) + 
    model.shadow_strength * torch.matmul(X_test_tensor, model.hidden_shadow),
    dim=1
).cpu().detach().numpy()

# Apply TSNE
tsne = TSNE(n_components=2, random_state=42)
tsne_without = tsne.fit_transform(hidden_without_shadow)
tsne_with = tsne.fit_transform(hidden_with_shadow)

# Plotting
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(tsne_without[:, 0], tsne_without[:, 1], c=y_test, cmap='viridis')
plt.title('Hidden Layer without Shadow Influence')

plt.subplot(1, 2, 2)
plt.scatter(tsne_with[:, 0], tsne_with[:, 1], c=y_test, cmap='viridis')
plt.title('Hidden Layer with Shadow Influence')

plt.show()


In [ ]:
import torch
import numpy as np

# Test input data
test = np.array([20, 30], dtype=np.float32)
test = (test - data.mean(axis=0).numpy()) / data.std(axis=0).numpy()
test_tensor = torch.tensor(test.reshape(1, 2))  # Reshape and convert to tensor

# Make prediction
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for inference
    prediction = model(test_tensor)

# Debug the shape of the output
print("Output shape:", prediction.shape)
print("Prediction:", prediction.flatten().numpy())  # Convert to a numpy array for clarity
